# StutterNet – Optimized Model

We optimize a trained **StutterNet (TDNN-based)** using **channel pruning** followed
by **INT8 TFLite quantization**, and measure the improvement in model size,
computation cost, and inference speed.

```
MyDrive/FluentSpeech/
├── clips/
├── SEP-28k_labels.csv
├── mfcc_cache.npz          ← from StutterNet_train.ipynb
└── models/
    ├── stutternet_base.pth ← from StutterNet_train.ipynb
    ├── stutternet_pruned.pth          (produced here)
    ├── stutternet_pruned.onnx         (produced here)
    ├── stutternet_tf/                 (produced here)
    └── stutternet_int8.tflite         (produced here)
```

---

## Design Choices

### Pruning — Channel Pruning (Structured), not Fine-grained (Unstructured)

| | Fine-grained pruning | **Channel pruning** ← chosen |
|---|---|---|
| What is removed | Individual weights | Entire output channels |
| Tensor shape after pruning | Unchanged (sparse weights) | **Smaller (dense tensors)** |
| TFLite INT8 Conv1d kernel | ✗ No sparse support | ✓ Native dense kernel |
| Real speedup on Android | ✗ | ✓ |

Channel pruning physically reduces tensor dimensions, yielding real MACs reduction
and real latency improvement on any hardware.  
We rank channels by their **L2 (Frobenius) norm**: a channel whose kernel has
large norm contributes more to the output → it is kept.  
Sorting channels by importance **before** truncation ensures maximum accuracy retention.

**Pruning scope:** connections between tdnn1→tdnn2→tdnn3→tdnn4.  
The tdnn5→fc1 boundary involves statistical pooling and is left intact.

### Quantization — INT8 Linear Quantization, not K-Means

| | K-Means quantization | **INT8 linear quantization** ← chosen |
|---|---|---|
| Weight representation | Codebook indices | Quantized integers |
| Inference kernel | Lookup table (slow) | **INT8 multiply-accumulate** |
| TFLite support | ✗ | ✓ Native INT8 ops |
| Android ARM speedup | ✗ | **~2× via SIMD** |
| Model size reduction | ×4 (2-bit) | **×4 (float32 → int8)** |

TFLite full-integer INT8 quantization maps both weights **and** activations to 8-bit
integers using a small representative calibration set.  
ARM Cortex cores (all Android phones) have dedicated INT8 SIMD instructions,
delivering real inference speedup with only a minor accuracy drop.


In [ ]:
# ── 1. Library Installation ─────────────────────────────────────────
!pip install -q "numpy<2.0" "protobuf==4.25.3"
!pip install -q librosa scikit-learn tqdm onnx matplotlib
!pip install -q torchprofile tensorflow onnx2tf onnx-graphsurgeon

In [2]:
# ── 2. Google Drive Mount ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ── 3. Imports & Settings ────────────────────────────────────────────
import copy, os, time, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm
import onnx
warnings.filterwarnings('ignore')

# ── Path Settings ──
DRIVE_ROOT        = '/content/drive/MyDrive/26-1_mobicom'
CACHE_PATH        = os.path.join(DRIVE_ROOT, 'mfcc_cache.npz')
BASE_MODEL_PATH   = os.path.join(DRIVE_ROOT, 'models', 'stutternet_base.pth')
PRUNED_PTH_PATH   = os.path.join(DRIVE_ROOT, 'models', 'stutternet_pruned.pth')
BASE_ONNX_PATH    = os.path.join(DRIVE_ROOT, 'models', 'stutternet_base.onnx')
PRUNED_ONNX_PATH  = os.path.join(DRIVE_ROOT, 'models', 'stutternet_pruned.onnx')
TF_SAVED_MODEL    = os.path.join(DRIVE_ROOT, 'models', 'stutternet_tf')
TFLITE_PATH       = os.path.join(DRIVE_ROOT, 'models', 'stutternet_int8.tflite')
os.makedirs(os.path.join(DRIVE_ROOT, 'models'), exist_ok=True)

# ── Audio / MFCC Settings (identical to training notebook) ──
SAMPLE_RATE = 16000
N_MFCC      = 20
HOP_LENGTH  = 160    # 10 ms @ 16 kHz
WIN_LENGTH  = 400    # 25 ms @ 16 kHz
N_FFT       = 512
MAX_FRAMES  = 300    # ~3 sec

# ── Training Settings ──
BATCH_SIZE    = 64
NUM_CLASSES   = 5
MAJORITY_VOTE = 2
STUTTER_COLS  = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection']

# ── Optimization Settings ──
PRUNE_RATIO     = 0.30   # Remove 30% of channels in tdnn1..tdnn4 inter-connections
FINETUNE_EPOCHS = 10
FINETUNE_LR     = 1e-4
CALIB_SAMPLES   = 500    # INT8 calibration samples

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Drive root exists: {os.path.exists(DRIVE_ROOT)}')

Device: cuda
Drive root exists: True


## 1. Model Definition & Data Loading

In [4]:
# ── 4. TDNN Layer & StutterNet (identical to training notebook) ──
class TDNNLayer(nn.Module):
    """1-D Dilated Conv + BN + ReLU (single TDNN layer)."""
    def __init__(self, in_dim, out_dim, context_size, dilation=1):
        super().__init__()
        padding = (context_size - 1) * dilation // 2
        self.conv = nn.Conv1d(in_dim, out_dim, kernel_size=context_size,
                              dilation=dilation, padding=padding)
        self.bn   = nn.BatchNorm1d(out_dim)

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)))


class StutterNet(nn.Module):
    """
    StutterNet (TDNN-based) for stuttering event classification.
    Input : (batch, MAX_FRAMES, N_MFCC)
    Output: (batch, NUM_CLASSES)  — raw logits (apply sigmoid for probabilities)
    """
    def __init__(self, input_dim=N_MFCC, num_classes=NUM_CLASSES):
        super().__init__()
        # TDNN Stack
        self.tdnn1 = TDNNLayer(input_dim, 512, context_size=5)
        self.tdnn2 = TDNNLayer(512,       512, context_size=5)
        self.tdnn3 = TDNNLayer(512,       512, context_size=7)
        self.tdnn4 = TDNNLayer(512,       512, context_size=1)
        self.tdnn5 = TDNNLayer(512,      1500, context_size=1)
        # Statistical Pooling + FC
        self.fc1     = nn.Linear(3000, 512)   # mean(1500) + std(1500)
        self.bn6     = nn.BatchNorm1d(512)
        self.fc2     = nn.Linear(512,  512)
        self.bn7     = nn.BatchNorm1d(512)
        self.output  = nn.Linear(512, num_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = x.transpose(1, 2)              # (B, N_MFCC, T)
        x = self.tdnn1(x); x = self.tdnn2(x); x = self.tdnn3(x)
        x = self.tdnn4(x); x = self.tdnn5(x)   # (B, 1500, T')
        # Statistical Pooling
        x = torch.cat([x.mean(-1), x.std(-1)], dim=-1)   # (B, 3000)
        x = F.relu(self.bn6(self.fc1(x))); x = self.dropout(x)
        x = F.relu(self.bn7(self.fc2(x))); x = self.dropout(x)
        return self.output(x)


# ── Load Base Model ──
ckpt  = torch.load(BASE_MODEL_PATH, map_location=DEVICE)
model = StutterNet().to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nBest Val F1   : {ckpt["val_f1"]:.4f}')
print(f'# of parameters: {total_params:,}')

StutterNet(
  (tdnn1): TDNNLayer(
    (conv): Conv1d(20, 512, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (tdnn2): TDNNLayer(
    (conv): Conv1d(512, 512, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (tdnn3): TDNNLayer(
    (conv): Conv1d(512, 512, kernel_size=(7,), stride=(1,), padding=(3,))
    (bn): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (tdnn4): TDNNLayer(
    (conv): Conv1d(512, 512, kernel_size=(1,), stride=(1,))
    (bn): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (tdnn5): TDNNLayer(
    (conv): Conv1d(512, 1500, kernel_size=(1,), stride=(1,))
    (bn): BatchNorm1d(1500, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (fc1): Linear(in_features=3000, out_features=512, bias=Tru

In [5]:
# ── 5. Load Cached MFCC Features ────────────────────────────────
cache    = np.load(CACHE_PATH)
features = cache['features']
labels   = cache['labels']
print(f'Cache loaded  features={features.shape}  labels={labels.shape}')

class StutterDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

idx = np.arange(len(features))
train_idx, test_idx  = train_test_split(idx, test_size=0.10, random_state=42)
train_idx, val_idx   = train_test_split(train_idx, test_size=0.11, random_state=42)

train_loader = DataLoader(StutterDataset(features[train_idx], labels[train_idx]),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(StutterDataset(features[val_idx],   labels[val_idx]),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(StutterDataset(features[test_idx],  labels[test_idx]),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_idx):>5d}')
print(f'Val  : {len(val_idx):>5d}')
print(f'Test : {len(test_idx):>5d}')

Cache loaded  features=(20131, 300, 20)  labels=(20131, 5)
Train: 16124
Val  :  1993
Test :  2014


In [6]:
# ── 6. Evaluation Function ───────────────────────────────────────
def evaluate(model, loader, threshold=0.5):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(DEVICE))
            preds  = (torch.sigmoid(logits) > threshold).cpu().numpy()
            preds_all.append(preds)
            labels_all.append(y.numpy())
    preds_all  = np.vstack(preds_all)
    labels_all = np.vstack(labels_all)
    macro_f1   = f1_score(labels_all, preds_all, average='macro', zero_division=0)
    return macro_f1, preds_all, labels_all


base_f1, _, _ = evaluate(model, test_loader)
print(f'=== Base Model Test Macro F1: {base_f1:.4f} ===')

=== Base Model Test Macro F1: 0.3608 ===


## 2. Channel Pruning

In [7]:
# ── 7. Channel Importance (Frobenius Norm) ───────────────────────
@torch.no_grad()
def get_output_channel_importance(weight):
    """
    Compute per-output-channel importance as Frobenius norm.
    weight shape : (out_ch, in_ch, kernel_size)
    Returns      : Tensor of shape (out_ch,)
    """
    return torch.stack([
        torch.norm(weight[i], p='fro') for i in range(weight.shape[0])
    ])

In [8]:
# ── 8. Sort Channels by Importance ──────────────────────────────
@torch.no_grad()
def apply_channel_sorting(model):
    """
    Re-order each TDNN layer's output channels in descending importance
    so channel_prune() can simply keep the first n_keep channels.
    Covers pairs: tdnn1->tdnn2, tdnn2->tdnn3, tdnn3->tdnn4.
    (tdnn4->tdnn5 is left unchanged; tdnn5 feeds statistical pooling.)
    """
    model = copy.deepcopy(model)
    pairs = [
        (model.tdnn1, model.tdnn2),
        (model.tdnn2, model.tdnn3),
        (model.tdnn3, model.tdnn4),
    ]
    for prev, nxt in pairs:
        importance = get_output_channel_importance(prev.conv.weight.data)
        idx        = torch.argsort(importance, descending=True)

        prev.conv.weight.data     = torch.index_select(prev.conv.weight.data,     0, idx)
        if prev.conv.bias is not None:
            prev.conv.bias.data   = torch.index_select(prev.conv.bias.data,       0, idx)
        prev.bn.weight.data       = torch.index_select(prev.bn.weight.data,       0, idx)
        prev.bn.bias.data         = torch.index_select(prev.bn.bias.data,         0, idx)
        prev.bn.running_mean.data = torch.index_select(prev.bn.running_mean.data, 0, idx)
        prev.bn.running_var.data  = torch.index_select(prev.bn.running_var.data,  0, idx)

        nxt.conv.weight.data      = torch.index_select(nxt.conv.weight.data,      1, idx)

    return model

In [9]:
# ── 9. Physical Channel Pruning ──────────────────────────────────
@torch.no_grad()
def channel_prune(model, prune_ratio):
    """
    Physically reduce tensor dimensions by keeping the first n_keep channels.
    Assumes channels are already sorted by importance (most important first).
    """
    model = copy.deepcopy(model)
    pairs = [
        (model.tdnn1, model.tdnn2),
        (model.tdnn2, model.tdnn3),
        (model.tdnn3, model.tdnn4),
    ]
    for prev, nxt in pairs:
        n_keep = int(prev.conv.out_channels * (1.0 - prune_ratio))
        keep   = torch.arange(n_keep, device=prev.conv.weight.device)

        # Shrink previous layer output channels
        prev.conv.weight.data     = torch.index_select(prev.conv.weight.data,     0, keep)
        prev.conv.out_channels    = n_keep
        if prev.conv.bias is not None:
            prev.conv.bias.data   = torch.index_select(prev.conv.bias.data,       0, keep)

        # Shrink previous BN
        prev.bn.weight.data       = torch.index_select(prev.bn.weight.data,       0, keep)
        prev.bn.bias.data         = torch.index_select(prev.bn.bias.data,         0, keep)
        prev.bn.running_mean.data = torch.index_select(prev.bn.running_mean.data, 0, keep)
        prev.bn.running_var.data  = torch.index_select(prev.bn.running_var.data,  0, keep)
        prev.bn.num_features      = n_keep

        # Shrink next layer input channels
        nxt.conv.weight.data      = torch.index_select(nxt.conv.weight.data,      1, keep)
        nxt.conv.in_channels      = n_keep

    return model

In [10]:
# ── 10. Apply Pruning & Compare ──────────────────────────────────
# Without sorting (naive pruning, for reference)
pruned_nosort  = channel_prune(model, PRUNE_RATIO)
f1_nosort, _, _= evaluate(pruned_nosort, test_loader)
print(f'Pruned (no sort)  Test F1: {f1_nosort:.4f}')

# With importance sorting (retains more accuracy)
sorted_model   = apply_channel_sorting(model)
pruned_model   = channel_prune(sorted_model, PRUNE_RATIO)
f1_pruned, _, _= evaluate(pruned_model, test_loader)
print(f'Pruned (sorted)   Test F1: {f1_pruned:.4f}')
print(f'Sorting benefit   : {f1_pruned - f1_nosort:+.4f}  F1 improvement')

Pruned (no sort)  Test F1: 0.1298
Pruned (sorted)   Test F1: 0.3010
Sorting benefit   : +0.1713  F1 improvement


## 3. Fine-tuning After Pruning

In [11]:
# ── 11. Loss / Optimizer / Scheduler ─────────────────────────────
def compute_pos_weight(label_array):
    """pos_weight for class imbalance — same as training notebook."""
    pos     = label_array.sum(axis=0)
    neg     = len(label_array) - pos
    weights = neg / (pos + 1e-8)
    weights[2] *= 0.2   # down-weight SoundRep
    weights[3] *= 0.3   # down-weight WordRep
    return torch.FloatTensor(weights)

pos_weight = compute_pos_weight(labels[train_idx]).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = torch.optim.Adam(pruned_model.parameters(),
                               lr=FINETUNE_LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINETUNE_EPOCHS)
print('pos_weight:', pos_weight.cpu().numpy().round(2))

pos_weight: [8.96 7.09 2.04 2.36 3.28]


In [12]:
# ── 12. Fine-tuning Loop ─────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


best_ft_f1 = 0.0
print(f'Fine-tuning for {FINETUNE_EPOCHS} epochs  (lr={FINETUNE_LR}) ...')

for epoch in range(1, FINETUNE_EPOCHS + 1):
    train_loss = train_epoch(pruned_model, train_loader, optimizer, criterion)
    val_f1, _, _ = evaluate(pruned_model, val_loader)
    scheduler.step()

    marker = ''
    if val_f1 > best_ft_f1:
        best_ft_f1 = val_f1
        torch.save({'epoch': epoch,
                    'model_state_dict': pruned_model.state_dict(),
                    'val_f1': val_f1},
                   PRUNED_PTH_PATH)
        marker = '  <- best saved'
    print(f'Epoch {epoch:02d}/{FINETUNE_EPOCHS}  Loss={train_loss:.4f}  Val F1={val_f1:.4f}{marker}')

# Reload best fine-tuned weights
pruned_model.load_state_dict(torch.load(PRUNED_PTH_PATH)['model_state_dict'])
pruned_f1, pruned_preds, pruned_labels = evaluate(pruned_model, test_loader)
print(f'\n=== Pruned + Fine-tuned  Test Macro F1: {pruned_f1:.4f} (Base: {base_f1:.4f}) ===')
print(classification_report(pruned_labels, pruned_preds,
                             target_names=STUTTER_COLS, zero_division=0))

Fine-tuning for 10 epochs  (lr=0.0001) ...
Epoch 01/10  Loss=0.2104  Val F1=0.3746  <- best saved
Epoch 02/10  Loss=0.1747  Val F1=0.3748  <- best saved
Epoch 03/10  Loss=0.1615  Val F1=0.3730
Epoch 04/10  Loss=0.1412  Val F1=0.3778  <- best saved
Epoch 05/10  Loss=0.1254  Val F1=0.3713
Epoch 06/10  Loss=0.1059  Val F1=0.3705
Epoch 07/10  Loss=0.0955  Val F1=0.3725
Epoch 08/10  Loss=0.0852  Val F1=0.3701
Epoch 09/10  Loss=0.0784  Val F1=0.3736
Epoch 10/10  Loss=0.0742  Val F1=0.3742

=== Pruned + Fine-tuned  Test Macro F1: 0.3451 (Base: 0.3608) ===
              precision    recall  f1-score   support

Prolongation       0.30      0.44      0.36       189
       Block       0.24      0.23      0.23       231
    SoundRep       0.27      0.19      0.22       185
     WordRep       0.27      0.31      0.29       240
Interjection       0.60      0.65      0.62       465

   micro avg       0.39      0.42      0.40      1310
   macro avg       0.34      0.36      0.35      1310
weighted av

## 4. ONNX Export

In [13]:
# ── 13. Export Base Model to ONNX ───────────────────────────────
dummy = torch.randn(1, MAX_FRAMES, N_MFCC).to(DEVICE)
model.eval()

torch.onnx.export(
    model, dummy, BASE_ONNX_PATH,
    export_params=True, opset_version=11, do_constant_folding=True,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    dynamo=False
)
onnx.checker.check_model(onnx.load(BASE_ONNX_PATH))
base_onnx_mb = os.path.getsize(BASE_ONNX_PATH) / (1024**2)
print(f'Base ONNX saved')
print(f'Path: {BASE_ONNX_PATH}')
print(f'Size: {base_onnx_mb:.1f} MB')

Base ONNX saved
Path: /content/drive/MyDrive/26-1_mobicom/models/stutternet_base.onnx
Size: 23.0 MB


In [14]:
# ── 14. Export Pruned Model to ONNX ───────────────
pruned_model.eval()
dummy = torch.randn(1, MAX_FRAMES, N_MFCC).to(DEVICE)

torch.onnx.export(
    pruned_model, dummy, PRUNED_ONNX_PATH,
    export_params=True, opset_version=11, do_constant_folding=True,
    input_names=['input'], output_names=['output'],
    dynamo=False
)
onnx.checker.check_model(onnx.load(PRUNED_ONNX_PATH))
pruned_onnx_mb = os.path.getsize(PRUNED_ONNX_PATH) / (1024**2)
print(f'Pruned ONNX saved  ({pruned_onnx_mb:.1f} MB)  static shape=(1,{MAX_FRAMES},{N_MFCC})')

Pruned ONNX saved  (16.5 MB)  static shape=(1,300,20)


## 5. INT8 TFLite Quantization

In [16]:
# ── 15. Install ONNX Quantization Package ────────────────────────
!pip install -q onnxruntime onnxruntime-extensions

import onnxruntime as ort
print(f'onnxruntime version: {ort.__version__}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 62.3 MB/s eta 0:00:00
onnxruntime version: 1.26.0


In [18]:
# ── 16. ONNX Static INT8 Quantization ────────────────────────────
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType
import onnxruntime as ort

INT8_ONNX_PATH = os.path.join(DRIVE_ROOT, 'models', 'stutternet_int8.onnx')

class MFCCCalibReader(CalibrationDataReader):
    def __init__(self, features, n_samples=CALIB_SAMPLES):
        idx        = np.random.choice(len(features), n_samples, replace=False)
        self.data  = [features[i:i+1].astype(np.float32) for i in idx]
        self.pos   = 0

    def get_next(self):
        if self.pos >= len(self.data):
            return None
        d = {'input': self.data[self.pos]}
        self.pos += 1
        return d

quantize_static(
    model_input=PRUNED_ONNX_PATH,
    model_output=INT8_ONNX_PATH,
    calibration_data_reader=MFCCCalibReader(features),
    weight_type=QuantType.QInt8
)

pruned_mb = os.path.getsize(PRUNED_ONNX_PATH) / (1024**2)
int8_mb   = os.path.getsize(INT8_ONNX_PATH)   / (1024**2)
print(f'Pruned ONNX (float32) : {pruned_mb:.1f} MB')
print(f'INT8   ONNX (static)  : {int8_mb:.1f} MB')
print(f'Size reduction        : {pruned_mb/int8_mb:.1f}x')

Pruned ONNX (float32) : 16.5 MB
INT8   ONNX (static)  : 4.2 MB
Size reduction        : 4.0x


In [19]:
# ── 17. Evaluate INT8 ONNX on Test Set ───────────────────────────
sess    = ort.InferenceSession(INT8_ONNX_PATH, providers=['CPUExecutionProvider'])
in_name = sess.get_inputs()[0].name
print(f'Input  : {sess.get_inputs()[0].name}  {sess.get_inputs()[0].shape}')
print(f'Output : {sess.get_outputs()[0].name}  {sess.get_outputs()[0].shape}')

preds_all, labels_all = [], []
for x, y in tqdm(test_loader, desc='INT8 ONNX Eval'):
    for i in range(len(x)):
        logit = sess.run(None, {in_name: x[i:i+1].numpy()})[0][0]
        prob  = 1.0 / (1.0 + np.exp(-logit))
        preds_all.append((prob > 0.5).astype(int))
        labels_all.append(y[i].numpy())

pa      = np.array(preds_all)
la      = np.array(labels_all)
int8_f1 = f1_score(la, pa, average='macro', zero_division=0)
print(f'\n=== INT8 ONNX  Test Macro F1: {int8_f1:.4f} ===')
print(classification_report(la, pa, target_names=STUTTER_COLS, zero_division=0))

Input  : input  [1, 300, 20]
Output : output  [1, 5]


INT8 ONNX Eval: 100%|██████████| 32/32 [00:42<00:00,  1.33s/it]


=== INT8 ONNX  Test Macro F1: 0.2982 ===
              precision    recall  f1-score   support

Prolongation       0.31      0.44      0.36       189
       Block       0.39      0.04      0.07       231
    SoundRep       0.24      0.15      0.18       185
     WordRep       0.18      0.75      0.29       240
Interjection       0.79      0.47      0.59       465

   micro avg       0.31      0.40      0.34      1310
   macro avg       0.38      0.37      0.30      1310
weighted avg       0.46      0.40      0.35      1310
 samples avg       0.20      0.21      0.19      1310



## 7. Profiling — Params / MACs / Latency / File Size

Simulates inference on a single input (1 × MAX_FRAMES × N_MFCC) to measure
the cost of each optimization step.

In [23]:
# ── 18. Profiling ─────────────────────────────────────────────────
import time
import onnx
from onnx import numpy_helper
from torchprofile import profile_macs

WARMUP = 10
RUNS   = 100

def measure_latency_pytorch(m, dummy_input, n_warmup=WARMUP, n_runs=RUNS):
    m_cpu = m.cpu().eval()
    d     = dummy_input.cpu()
    with torch.no_grad():
        for _ in range(n_warmup): m_cpu(d)
        t0 = time.perf_counter()
        for _ in range(n_runs):   m_cpu(d)
    return (time.perf_counter() - t0) / n_runs * 1000  # ms

def measure_latency_onnx(sess, sample_np, n_warmup=WARMUP, n_runs=RUNS):
    in_name_ = sess.get_inputs()[0].name
    for _ in range(n_warmup):
        sess.run(None, {in_name_: sample_np})
    t0 = time.perf_counter()
    for _ in range(n_runs):
        sess.run(None, {in_name_: sample_np})
    return (time.perf_counter() - t0) / n_runs * 1000  # ms

dummy_single = torch.randn(1, MAX_FRAMES, N_MFCC)
dummy_np     = dummy_single.numpy()

lat_base   = measure_latency_pytorch(model,        dummy_single)
lat_pruned = measure_latency_pytorch(pruned_model, dummy_single)
lat_int8   = measure_latency_onnx(sess,            dummy_np)

# Move models back to GPU
model        = model.to(DEVICE)
pruned_model = pruned_model.to(DEVICE)

# MACs & Params (PyTorch)
dummy_gpu     = dummy_single.to(DEVICE)
macs_base     = profile_macs(model,        dummy_gpu)
macs_pruned   = profile_macs(pruned_model, dummy_gpu)
params_base   = sum(p.numel() for p in model.parameters())
params_pruned = sum(p.numel() for p in pruned_model.parameters())

# MACs & Params (INT8 ONNX)
macs_int8   = macs_pruned
int8_model  = onnx.load(INT8_ONNX_PATH)
params_int8 = sum(
    numpy_helper.to_array(init).size
    for init in int8_model.graph.initializer
)

# File sizes
base_pth_mb   = os.path.getsize(BASE_MODEL_PATH)  / (1024**2)
pruned_pth_mb = os.path.getsize(PRUNED_PTH_PATH)  / (1024**2)
pruned_onnx_mb= os.path.getsize(PRUNED_ONNX_PATH) / (1024**2)
int8_onnx_mb  = os.path.getsize(INT8_ONNX_PATH)   / (1024**2)

# Print table
FMT = '{:<30} {:>9} {:>10} {:>10} {:>9} {:>10}'
SEP = '=' * 82
print(SEP)
print(FMT.format('Model', 'Test F1', 'Params', 'MACs (G)', 'Size', 'Lat (ms)'))
print('-' * 82)
print(FMT.format('Base  (.pth  float32)',
                 f'{base_f1:.4f}',
                 f'{params_base/1e6:.2f} M',
                 f'{macs_base/1e9:.4f}',
                 f'{base_pth_mb:.1f} MB',
                 f'{lat_base:.2f}'))
print(FMT.format('Pruned(.pth  float32)',
                 f'{pruned_f1:.4f}',
                 f'{params_pruned/1e6:.2f} M',
                 f'{macs_pruned/1e9:.4f}',
                 f'{pruned_pth_mb:.1f} MB',
                 f'{lat_pruned:.2f}'))
print(FMT.format('INT8  (.onnx int8)',
                 f'{int8_f1:.4f}',
                 f'{params_int8/1e6:.2f} M',
                 f'{macs_int8/1e9:.4f}',
                 f'{int8_onnx_mb:.1f} MB',
                 f'{lat_int8:.2f}'))
print(SEP)
print(f'MACs reduction   (pruning)      : {macs_base/macs_pruned:.1f}x')
print(f'Param reduction  (pruning)      : {params_base/params_pruned:.1f}x')
print(f'Latency speedup  (pruned pth)   : {lat_base/lat_pruned:.1f}x')
print(f'Latency speedup  (int8 onnx)    : {lat_base/lat_int8:.1f}x')
print(f'Size reduction   (pth -> int8)  : {base_pth_mb/int8_onnx_mb:.1f}x')
print(f'F1 drop (pruning)               : {pruned_f1 - base_f1:+.4f}')
print(f'F1 drop (INT8 quantization)     : {int8_f1   - base_f1:+.4f}')
print()
print(f'Android deployment target:')
print(f'  {INT8_ONNX_PATH}')
print( '  -> use ONNX Runtime Mobile on Android')

Model                            Test F1     Params   MACs (G)      Size   Lat (ms)
----------------------------------------------------------------------------------
Base  (.pth  float32)             0.3608     6.04 M     1.2710   23.1 MB      32.73
Pruned(.pth  float32)             0.3451     4.34 M     0.7602   16.6 MB      22.63
INT8  (.onnx int8)                0.2982     4.33 M     0.7602    4.2 MB      13.46
MACs reduction   (pruning)      : 1.7x
Param reduction  (pruning)      : 1.4x
Latency speedup  (pruned pth)   : 1.4x
Latency speedup  (int8 onnx)    : 2.4x
Size reduction   (pth -> int8)  : 5.5x
F1 drop (pruning)               : -0.0157
F1 drop (INT8 quantization)     : -0.0626

Android deployment target:
  /content/drive/MyDrive/26-1_mobicom/models/stutternet_int8.onnx
  -> use ONNX Runtime Mobile on Android
